# 01 — Exploratory Data Analysis

IEEE-CIS Fraud Detection dataset (Vesta Corporation).  
Goal: understand class imbalance, temporal structure, missingness, and key feature distributions before modelling.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src importable from notebooks/

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

from src.data.load import load_raw

df = load_raw()
print(f'Shape: {df.shape}  |  Fraud rows: {df.isFraud.sum():,}  |  Fraud rate: {df.isFraud.mean():.4%}')

## 1. Class Balance

In [ ]:
counts = df['isFraud'].value_counts()
labels = ['Legitimate', 'Fraud']
colors = ['#4C8EDA', '#E05C5C']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].pie(counts, labels=labels, colors=colors, autopct='%1.2f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Transaction Class Distribution')

axes[1].bar(labels, counts.values, color=colors, edgecolor='white')
axes[1].set_ylabel('Count')
axes[1].set_title('Raw Counts (note log scale)')
axes[1].set_yscale('log')
for bar, val in zip(axes[1].patches, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print(f'Class ratio (legitimate:fraud) = {counts[0]/counts[1]:.0f}:1')

## 2. Temporal Distribution

In [ ]:
# TransactionDT is seconds elapsed since an undisclosed reference epoch
df['tx_day'] = df['TransactionDT'] // 86400

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

daily = df.groupby('tx_day')['TransactionID'].count()
axes[0].fill_between(daily.index, daily.values, alpha=0.7, color='#4C8EDA')
axes[0].set_ylabel('Daily transactions')
axes[0].set_title('Transaction Volume over Time')

daily_fraud = df[df.isFraud == 1].groupby('tx_day')['TransactionID'].count()
daily_fraud_rate = (daily_fraud / daily).fillna(0)
axes[1].fill_between(daily_fraud_rate.index, daily_fraud_rate.values * 100,
                     alpha=0.7, color='#E05C5C')
axes[1].set_ylabel('Daily fraud rate (%)')
axes[1].set_xlabel('Day offset from dataset start')
axes[1].set_title('Daily Fraud Rate')

# Mark the 80/20 train/val split boundary
split_day = int(df['tx_day'].quantile(0.8))
for ax in axes:
    ax.axvline(split_day, color='black', linestyle='--', linewidth=1.2,
               label=f'Train/Val split (day {split_day})')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Date range: day 0 to day {df["tx_day"].max()}')
print(f'Train/val boundary: ~day {split_day}')

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for fraud_flag, label, color in [(0, 'Legitimate', '#4C8EDA'), (1, 'Fraud', '#E05C5C')]:
    subset = df.loc[df.isFraud == fraud_flag, 'TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=60, alpha=0.6, label=label, color=color, density=True)

axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Density')
axes[0].set_title('Amount Distribution (log scale)')
axes[0].legend()

df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[1],
           showfliers=False, patch_artist=True,
           boxprops={'facecolor': '#4C8EDA', 'alpha': 0.7})
axes[1].set_title('Amount by Class (no outliers)')
axes[1].set_xlabel('isFraud')
axes[1].set_ylabel('TransactionAmt ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()

for flag, label in [(0, 'Legitimate'), (1, 'Fraud')]:
    s = df.loc[df.isFraud == flag, 'TransactionAmt']
    print(f'{label:12s}  median=${s.median():7.2f}  mean=${s.mean():8.2f}  p95=${s.quantile(0.95):8.2f}')

## 4. Missingness Analysis

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
top_missing = null_pct[null_pct > 0].head(30)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_missing.index[::-1], top_missing.values[::-1] * 100,
               color='#E8A838', edgecolor='white')
ax.set_xlabel('% missing')
ax.set_title('Top 30 Columns by Missingness Rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(80, color='red', linestyle='--', linewidth=1, label='80% threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 columns by missingness:')
print(null_pct.head(10).map(lambda x: f'{x:.1%}').to_string())
print(f'\nColumns with >80% missing: {(null_pct > 0.8).sum()}')
print(f'Columns with any missing:  {(null_pct > 0).sum()} / {len(null_pct)}')

## 5. Categorical Feature Fraud Rates

In [ ]:
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.flat, cat_cols):
    rates = (
        df.groupby(col)['isFraud']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'fraud_rate', 'count': 'n'})
        .sort_values('fraud_rate', ascending=False)
        .head(15)
    )
    bars = ax.bar(range(len(rates)), rates['fraud_rate'] * 100,
                  color='#E05C5C', alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=35, ha='right', fontsize=8)
    ax.set_title(f'{col} — fraud rate by category')
    ax.set_ylabel('Fraud rate (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 6. V-Feature Correlations with isFraud

In [ ]:
v_cols = [c for c in df.columns if c.startswith('V')]
corr = df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
top_v = corr.abs().nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E05C5C' if corr[c] < 0 else '#4C8EDA' for c in top_v.index]
ax.barh(top_v.index[::-1], corr[top_v.index[::-1]], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Pearson correlation with isFraud')
ax.set_title('Top 20 V-Features by |Correlation| with isFraud\n(blue=positive, red=negative)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(f'V-features with |corr| > 0.05: {(corr.abs() > 0.05).sum()}')

## 7. Identity Table Join Rate

In [ ]:
id_cols = [c for c in df.columns if c.startswith('id_') or c in ('DeviceType', 'DeviceInfo')]
joined = df[id_cols[0]].notna().mean()

print(f'Identity match rate: {joined:.1%} of transactions have identity data')
print(f'  → {df[id_cols[0]].notna().sum():,} rows with identity records')
print(f'  → {df[id_cols[0]].isna().sum():,} rows without identity records')

# Fraud rate for matched vs unmatched
for has_id, label in [(True, 'Has identity'), (False, 'No identity')]:
    mask = df[id_cols[0]].notna() == has_id
    rate = df.loc[mask, 'isFraud'].mean()
    print(f'  {label}: fraud rate = {rate:.3%}')

## 8. Summary

| Finding | Value |
|---------|-------|
| Total training transactions | ~590,540 |
| Fraud rate | ~3.5% |
| Class imbalance ratio | ~28:1 |
| Features (post-join) | 434 |
| Columns with >80% missing | see cell 4 |
| V-features with \|corr\|>0.05 | see cell 6 |

**Key modelling implications:**
- Heavy class imbalance → use `scale_pos_weight` in XGBoost or AUC-PR as primary metric.
- XGBoost handles NaN natively — no imputation needed, preserving missingness as a signal.
- Time-based split is mandatory: temporal autocorrelation would inflate random-split CV scores.
- Identity match rate (~15–20%) means identity features are high-signal but sparse; tree models handle this well.